In [ ]:
## critical for running the tutorial on jupyter notebook
## ignore if running on terminal
import nest_asyncio2  # type: ignore[import-untyped]

nest_asyncio2.apply()

# World Model

> **User Problem**.  
> A resident living in our smart home wants to ask the AI assistant questions about their home, such as *"which devices in my kitchen are  currently running?"*, so that they can quickly monitor and manage their home without needing to remember explicit device names or manually inspect individual rooms. 

In this tutorial, we will build this functionality in our smart home assistant. 

Until now, we have described the environment using a flat representation. For example, a `Device` named `bedroom` or `main_lock`. This simple setup works when users issue explicit commands like *"lock the main lock"*. However, it breaks down when handling complex, relational queries, such as *"Which devices in the kitchen are currently running?"* 

To reason effectively, an agent needs more than isolated identifiers; it requires a **world model** an internal, structured representation of its physical and conceptual environment that captures how entities relate to one another and change over time.

`cognition` uses a graph data structure to power an agent's world model. By organizing knowledge into a graph data structure, `cognition` provides the structural context needed to parse and resolve complex spatial and relational queries. As we will see later, this graph-based world model is not just a storage system, but the core computational substrate for agent reasoning, planning, and decision making.

## Step 1: Defining the Data Models
In `cognition`, a world model is built on two core concepts: **Entities** and **Relations** i.e, nodes and edges in a graph. 

- **Entities** (*nodes*): Represent distinct objects, spaces, or concepts within the world. Every entity requires a unique name at the time of instantiation to act as its identifier across the graph. Entities can have two types of attributes. 
    - Immutable attributes are properties set at creation, typically used to set affordances, i.e, what actions can be performed on the entity. For example, a dishwasher’s affordances (`DeviceType.RUNNABLE` and `DeviceType.FILLABLE`) remain fixed throughout its lifecycle. 
    - Mutable attributes are used typically to capture dynamic state that change during the agent's lifecycle. For examples, transitioning a device's operational state from `DeviceState.IDLE` to `DeviceState.RUNNING`.
- **Relations** (*edges*): Connections defined over entities. Relations bind unique entities together into a structure network, capturing spatial, hierarchical, or operational context (such as `Device("bedroom_lamp") -- [IN] --> Room("room:bed")`).

Let's define the world model for our smart home assistant.

In [ ]:
from enum import Enum, StrEnum, auto
from pydantic import Field
from cognition import Entity, BinaryRelation


#-------------------------------------------------------------------
# 1. Attributes: Affordances (Capabilities) & States
#-------------------------------------------------------------------


class DeviceType(Enum):
    """Immutable affordances defining what an entity can do."""

    LIGHTABLE = auto()
    LOCKABLE = auto()
    RUNNABLE = auto()
    MOVABLE = auto()
    FILLABLE = auto()


class DeviceState(StrEnum):
    """Mutable states reflecting current operational conditions."""

    ON = "on"
    OFF = "off"
    LOCKED = "locked"
    UNLOCKED = "unlocked"
    IDLE = "idle"
    RUNNING = "running"
    EMPTY = "empty"
    FULL = "full"


class DeviceName(StrEnum):
    """Standardized identifiers for known entities."""

    BEDROOM = "bedroom"
    LIVING_ROOM = "living_room"
    KITCHEN = "kitchen"
    BATHROOM = "bathroom"
    MAIN_DOOR = "main_door"
    DISHWASHER = "dishwasher"
    WASHER = "washer"
    LAUNDRY_BASKET = "laundry_basket"


# -------------------------------------------------------------------
# 2. Entities (Require a Unique Name upon Instantiation)
# -------------------------------------------------------------------


class Device(Entity):
    """Smart home device entity combining immutable affordances and mutable state.

    Inherits `name` from Entity, which must be uniquely specified at instantiation.
    """

    dtype: frozenset[DeviceType]  # Immutable affordances
    dstate: DeviceState  # Mutable state


class Room(Entity):
    """Room entity.

    Requires a unique `name` string upon instantiation (e.g., Room(name="room:kitchen")).
    """


class RoomName(StrEnum):
    """Standardized identifiers for room nodes."""

    KITCHEN = "room:kitchen"
    LIVING_ROOM = "room:living"
    BEDROOM = "room:bed"
    BATHROOM = "room:bath"
    LAUNDRY = "room:laundry"


# -------------------------------------------------------------------
# 3. Relations (Defined Over Named Entities)
# -------------------------------------------------------------------
class In(BinaryRelation):
    """Directed relation linking a Device entity to a Room entity."""

    entity1: Device = Field(frozen=True)
    entity2: Room = Field(frozen=True)

## Step 2: Constructing the World Graph 

Once data models are established, the next step is to assemble them into an active graph. `cognition` provides a base `WorldGraph` class to encapsulate the topology of an agent's environment.

### Thinking at the Knowledge Level
A major architectural benefit of `cognition`'s `WorldGraph` is its knowledge-level abstraction. Rather than managing raw graph primitives such as vertex IDs, adjacency lists, or generic `add_node(id=1)` and `add_edge(1, 2)` calls, `WorldGraph` enables you operate directly at the knowledge-level:

- **Domain-Typed Entities & Relations**: You interact directly with high-level domain concepts (`Device`, `Room`, `In`) rather than low-level nodes and edges.

- **Semantic Safety**: Relations are enforced over validated domain objects (e.g., `In(entity1=washer, entity2=laundry)`). This prevents invalid edges and ensures every graph connection carries explicit semantic meaning.

- **Declarative Construction**: Agent designers focus on describing what the world contains (affordances, states, locations) rather than managing underlying graph indexes and pointers.

In [ ]:
from cognition import WorldGraph

def init_house_state() -> WorldGraph:
    """initialize a world graph corresponding to the smart home environment"""

    world = WorldGraph()
    # ---------------------------------------------------------------
    # 1. Bedroom Topology
    # ---------------------------------------------------------------
    bedroom = Room(name=RoomName.BEDROOM)
    bedroom_lamp = Device(
        name="bedroom",
        dtype=frozenset([DeviceType.LIGHTABLE]),
        dstate=DeviceState.ON,
    )
    world.add_entity(bedroom)
    world.add_entity(bedroom_lamp)
    world.add_relation(In(entity1=bedroom_lamp, entity2=bedroom))

    # ---------------------------------------------------------------
    # 2. Living Room Topology
    # ---------------------------------------------------------------
    living_room = Room(name=RoomName.LIVING_ROOM)
    living_room_lamp = Device(
        name="living_room",
        dtype=frozenset([DeviceType.LIGHTABLE]),
        dstate=DeviceState.ON,
    )
    main_door = Device(
        name="main_door",
        dtype=frozenset([DeviceType.LOCKABLE]),
        dstate=DeviceState.LOCKED,
    )
    world.add_entity(living_room)
    world.add_entity(living_room_lamp)
    world.add_entity(main_door)
    world.add_relation(In(entity1=living_room_lamp, entity2=living_room))
    world.add_relation(In(entity1=main_door, entity2=living_room))

    # ---------------------------------------------------------------
    # 3. Bathroom Topology
    # ---------------------------------------------------------------
    bathroom = Room(name=RoomName.BATHROOM)
    bathroom_lamp = Device(
        name="bathroom", dtype=frozenset([DeviceType.LIGHTABLE]), dstate=DeviceState.ON
    )
    world.add_entity(bathroom)
    world.add_entity(bathroom_lamp)
    world.add_relation(In(entity1=bathroom_lamp, entity2=bathroom))

    # ---------------------------------------------------------------
    # 4. Kitchen Topology
    # ---------------------------------------------------------------
    kitchen = Room(name=RoomName.KITCHEN)
    kitchen_lamp = Device(
        name="kitchen", dtype=frozenset([DeviceType.LIGHTABLE]), dstate=DeviceState.ON
    )

    dishwasher = Device(
        name="dishwasher",
        dtype=frozenset([DeviceType.RUNNABLE, DeviceType.FILLABLE]),
        dstate=DeviceState.RUNNING,
    )
    world.add_entity(kitchen)
    world.add_entity(kitchen_lamp)
    world.add_entity(dishwasher)
    world.add_relation(In(entity1=kitchen_lamp, entity2=kitchen))
    world.add_relation(In(entity1=dishwasher, entity2=kitchen))

    # ---------------------------------------------------------------
    # 5. Laundry Room Topology
    # ---------------------------------------------------------------
    laundry = Room(name=RoomName.LAUNDRY)
    washer = Device(
        name="washer",
        dtype=frozenset([DeviceType.RUNNABLE, DeviceType.FILLABLE]),
        dstate=DeviceState.IDLE,
    )
    dryer = Device(
        name="dryer",
        dtype=frozenset([DeviceType.RUNNABLE, DeviceType.FILLABLE]),
        dstate=DeviceState.IDLE,
    )
    laundry_basket = Device(
        name="laundry_basket",
        dtype=frozenset([DeviceType.MOVABLE, DeviceType.FILLABLE]),
        dstate=DeviceState.EMPTY,
    )
    world.add_entity(laundry)
    world.add_entity(washer)
    world.add_entity(dryer)
    world.add_entity(laundry_basket)
    world.add_relation(In(entity1=washer, entity2=laundry))
    world.add_relation(In(entity1=dryer, entity2=laundry))
    world.add_relation(In(entity1=laundry_basket, entity2=laundry))

    return world


In [ ]:
world = init_house_state()

## Step 3: Inspecting Graph State (`world.snapshot`)
At any point, you can inspect the current state of the world model using `WorldGraph`'s snapshot property. `world.snapshot` generates an immutable, point-in-time view of all registered entities and relations as a set of active *facts*.

In [ ]:
# Inspect the point-in-time facts of the world model
print(world.snapshot)

### Dual-Mode Access to World Model: Dynamic Tracking v/s Stable Reasoning
Notice that `Device`, `Room`, and `In` instances in `world.snapshot` are automatically cast to `FrozenDevice`, `FrozenRoom`, and `FrozenIn`.

In `cognition`, the world model explicitly supports dual-mode access to reconcile dynamic state tracking with stable cognitive processing:

- The Mutable Graph (Dynamic State Tracking): Environments are continuously changing. As sensors poll new data, user inputs arrive, or agents execute actions, the underlying `WorldGraph` mutates in real time to reflect the latest physical reality.

- Immutable Snapshots (Stable, Consistent Reasoning): Deductive reasoning, multi-step planning, and LLM reasoning take computational time. If the underlying world state mutates while an agent is mid-thought, its reasoning can become inconsistent, lead to invalid plans, or trigger race conditions.

By generating read-only, frozen snapshots, cognition decouples environmental updates from cognitive deliberation.

## Step 4: Querying the World Model

Once the world model is constructed, an agent must be able to inspect and reason over it. Rather than requiring developers to write fragile string-parsing heuristics or nested loops, `cognition` provides a unified, declarative query interface through `world.snapshot.by()`.

The `.by()` method enables both node-level queries (filtering entities by type or property) and edge-level traversals (filtering relations across connected entities).

> Note: Working with Frozen Subclasses. 
>
> When querying `world.snapshot`, the returned items are read-only frozen subclasses of your defined domain classes (e.g., `FrozenDevice`, `FrozenRoom`, `FrozenIn` derived from `Device`, `Room`, and `In`). These immutable subclasses guarantee that query execution and agent reasoning loops cannot accidentally mutate the underlying world state. Because frozen subclasses inherit directly from your base domain models, standard type checks like `isinstance(in_rel.entity1, Device)` continue to evaluate as expected.

### Example A: Fetching All Entities of a Specific Type 

To retrieve every node belonging to a given entity class, pass the entity class directly to `.by()`:

In [ ]:
# Retrieve all Room entities in the world graph
facts = world.snapshot.by(Room)
fact_list = list(facts)
print(*fact_list, sep="\n")

### Example B: Filtering Entities by Attribute

To locate specific nodes, pass a predicate (`lambda`) function as the second argument. The method evaluates the predicate against each node of that entity type:

In [ ]:
# Filter rooms by name attribute
facts = world.snapshot.by(Room, lambda room: room.name == RoomName.KITCHEN)
fact_list = list(facts)
print(*fact_list, sep="\n")

### Example C: Relational Traversal Across Edges

To answer complex relational or spatial queries such as *"Which devices are currently in the kitchen?*, pass the target relation type (`In`) alongside a predicate function that evaluates the connected entities (`entity1` and `entity2`):

In [ ]:
# Find all devices in room:kitchen that are currently running
facts = world.snapshot.by(
    In,
    lambda in_rel: (
        in_rel.entity1.dstate == DeviceState.RUNNING
        and in_rel.entity2.name == RoomName.KITCHEN
    )
)
fact_list = list(facts)
print(*fact_list, sep="\n")

### Lazy Evaluation via Fact Generators
A key feature of `world.snapshot.by()` is lazy evaluation. When you pass a predicate function to filter facts, `.by()` does not allocate a new list in memory. Instead, it returns a Python generator that evaluates and yields facts satisfying your constraints on demand.

This generator-based design offers several operational advantages:

- **Memory Efficiency**: Large world graphs containing thousands of entities and spatial edges can be queried without allocating temporary lists for intermediate results.
- **Early Exit Optimization**: In decision-making routines (e.g., verifying if at least one device satisfies a precondition), agents can exit iteration immediately upon finding the first match `(next(facts))`, saving execution time.
- **Streamable Pipelines**: Filtered generators can be piped directly into downstream decision processes, scoring functions, or LLM context formatters.

## Step 5: Generating Natural Language Responses
While extracting raw facts like `FrozenIn(...)` is ideal for programmatic decision-making and reasoning, human users expect clear, natural language answers.

`cognition` provides the `describe_facts()` helper function to bridge formal world model queries with user-facing communication. By passing the filtered graph facts along with the user's intent to a language model via Pydantic's abstractions, `cognition` synthesizes an accurate, grounded response that directly answers the inquiry without hallucinating outside the world state.

### Example
The following example takes the fact generator from Step 4, collects the matching facts into a list, and passes them to `describe_facts()` alongside a target task description:

In [ ]:
from cognition import describe_facts
from pydantic_ai.models import infer_model
import os

assert "OPENAI_API_KEY" in os.environ, "Environment variable OPENAI_API_KEY is not set."
llm_model = infer_model("openai:gpt-4o")

response = describe_facts(fact_list, task_desc="resident question: which devices in my kitchen are running?", llm=llm_model)

print(response)


### Key Mechanics

- **Grounded Fact Synthesis**: `describe_facts()` formats the supplied `FrozenIn` and `FrozenDevice` objects into a structured context window for the LLM. The model is explicitly constrained to state facts present in the graph snapshot, preventing state hallucinations.

- **Intent Alignment**: The `task_desc` argument instructs the language model on how to frame the response so that it directly answers the resident's question rather than dumping raw property key-value pairs.

- **Pluggable LLM Backends**: Powered by `pydantic_ai.models`, `infer_model` allows you to swap model providers (e.g., OpenAI, Anthropic, or local open-source models) seamlessly without altering your graph query or fact-formatting logic.

## Step 6: Embedding World Model in Cogent's Decision Processes

In Step 5, we manually queried a static snapshot and formatted facts using an LLM. In an active agent architecture, the `WorldGraph` is attached directly to the agent's state definition (`HomeState`). This provides a persistent, internal representation of physical reality that the agent continuously updates and queries through a cogent's operators. 

Crucially, such a world model is essential in [**partially observable environments**](https://en.wikipedia.org/wiki/Partially_observable_system). Real-world sensors are rarely perfect: they may cover only a fraction of the space at any given moment, drop network connection, or stream incomplete data. By maintaining a persistent graph of entities and relations, the cogent retains structural memory of the world. This allows it to infer unobserved states from relational context, maintain an accurate snapshot of unchanged entities, and continue reasoning reliably even when live telemetry drops or degrades.

For a complete reference implementation, see [smart_home_assistant_v5.py](smart_home_assistant_v5.py).

### Dynamic State Synchronization (`UpdateWorldModel`)
A cogent's internal world model should be updated when external conditions change, such as a user flipping a phyiscal switch. We build a dedicated `Operator`, `UpdateWorldModel` that continuously compares incoming telemetry (`io.i.devices`) with the agent's internal `WorldGraph` (`state.world`) and mutates entities when discrepancies arise.

For a full reference implementation see [smart_home_assistant_v5.py](smart_home_assistant_v5.py).

In [ ]:
from typing import Any, cast
from cognition import Operator, IOContainer
from smart_home_assistant_v5 import HomeState, Observation, Device, DeviceState

class UpdateWorldModel(Operator[HomeState]):
    """WHEN: Sensor values differ from the internal world model
       THEN: Update the internal world graph to reflect external reality.
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._to_update: dict[str, Any] = {}

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        """Senses external device state changes and queues required updates."""
        observation = cast(Observation, io.i.devices)
        for device_name, raw_state in observation.model_dump().items():
            mydevice: Device = cast(Device, state.world.get_entity(device_name, linked=False))
            if mydevice.dstate != raw_state:
                self._to_update[device_name] = raw_state
        return len(self._to_update) > 0

    def perform(self, state: HomeState, io: IOContainer) -> None:
        """Mutates the live WorldGraph to synchronize internal state with real-world changes."""
        for device_name, raw_state in self._to_update.items():
            mydevice = state.world.get_entity(device_name, linked=True)
            print(f"--> [Update internal world model] -> setting {mydevice.e.name} to {DeviceState(raw_state)}")
            mydevice.e.dstate = DeviceState(raw_state)
            mydevice.update()
        self._to_update.clear()

### Mutating the Graph (`linked=True` VS `linked=False`)
Internally, the `WorldGraph` stores your data entities (like `Device` and `Room`) inside graph nodes that maintain relational configurations. When retrieving an entity using `state.world.get_entity()`, the `linked` parameter controls what the system returns:

- `linked=False` (**Raw Entity**): Returns the bare entity itself (e.g., the `Device` object). In the `can_perform` method above, we use `linked=False` for property checks when we don't need to mutate the graph.

- `linked=True` (**Linked Node - Default**: Returns the active node wrapper containing the entity. In the `perform` method, we need to mutated the graph. The linked node wrapper exposes the underlying entity via `.e` (allowing `mydevice.e.dstate=...`) and provides an `.update()` method to safely commit those changes back into the graph's internal indexes. 

### Live Execution

Run the complete smart home assistant script and interact with the physical simulation by toggling device switches:

In [ ]:
!python smart_home_assistant_v5.py

As external environment state changes occur, `UpdateWorldModel` automatically catches telemetry updates, logging mutations and keeping the agent's graph-based world model completely aligned with real-world conditions.

## Step 7: Responding to the Resident's Queries 

With the world model actively synchronizing external state changes (as shown in Step 6), the cogent can now answer natural language inquiries from the resident.

To handle resident queries end-to-end, `cognition` breaks the process down into three distinct, modular components: defining a query schema, extracting intent from natural language, and evaluating facts against the world model snapshot. For a full reference implementation see [smart_home_assistant_v5.py](smart_home_assistant_v5.py).

### Component 1: The Query Schema (`DeviceQuery`)
First, we define a Pydantic data model that serves as a structured target for natural language parsing. This schema captures optional filter fields (`dtype`, `dstate`, `room`) that correspond directly to entity attributes and relational edges in our world model.

In [ ]:
from pydantic import BaseModel
from smart_home_assistant_v5 import DeviceType, DeviceState, RoomName

class DeviceQuery(BaseModel):
    """Target parameters extracted from a resident's natural language query."""
    dtype: DeviceType | None = None
    dstate: DeviceState | None = None
    room: RoomName | None = None

### Component 2: The Intent Extraction Operator (`ExtractIntentDeviceQuery`)

Next, we create an operator responsible for converting unstructured user utterances into a populated `DeviceQuery` object. It uses `cognition`'s `ModelPopulator` to instruct an LLM to map raw text into our Pydantic schema. This operator is attached to the decision process defined in `ProcessHumanIntent` operator, extending the types of intents our cogent can respond to.

In [ ]:
from typing import Any, cast
from cognition import Operator, IOContainer, ModelPopulator
from smart_home_assistant_v5 import StructuredIntent, ResidentIntent, Utterance, KState, llm_model

class ExtractIntentDeviceQuery(Operator[StructuredIntent]):
    """
    Super-process operator: ProcessHumanIntent
    INIT: Instantiate ModelPopulator for DeviceQuery
    WHEN: Intent is ResidentIntent.GET_DEVICE_INFO and state.object is None
    THEN: Interpret user utterance as a DeviceQuery (or KState.UNKNOWN if unresolvable)
    """

    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(name, **kwargs)
        self._interpreter = ModelPopulator(
            DeviceQuery,
            task_desc="help the resident understand the state of devices in their rooms"
        )

    def can_perform(self, state: StructuredIntent, io: IOContainer) -> bool:
        return (
            state.intent is ResidentIntent.GET_DEVICE_INFO
            and not state.object
        )

    def perform(self, state: StructuredIntent, io: IOContainer) -> None:
        utterance = cast(Utterance, io.a.utterance)
        query = self._interpreter(utterance.phrase, llm_model)
        state.object = query if query else KState.UNKNOWN
        print(
            f"--> [Process intent decision process] -> recognized query {state.object}"
        )

### Component 3: The Graph Reasoning and Response Operator (`GenerateDeviceQueryResponse`)

Finally, this operator executes the actual graph reasoning. It inspects the populated `DeviceQuery`, constructs a dynamic predicate over `world.snapshot.by()`, retrieves matching facts, and uses `describe_facts()` to generate a natural language answer grounded strictly in the world model.

In [ ]:
from typing import cast
from cognition import Operator, IOContainer, describe_facts
from smart_home_assistant_v5 import HomeState, ResidentIntent, Device, In, llm_model, Utterance

class GenerateDeviceQueryResponse(Operator[HomeState]):
    """
    WHEN: Human intent has been parsed as a device query
    THEN: Execute device query against world graph snapshot and generate NL response
    """

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        if state.human_intent:
            return (
                state.human_intent.intent is ResidentIntent.GET_DEVICE_INFO
                and state.human_intent.object is not None
            )
        return False

    def perform(self, state: HomeState, io: IOContainer) -> HomeState | None:
        query = cast(DeviceQuery, state.human_intent.object)
        print("--> [Generate response to device query triggered]")

        # Execute predicate search over immutable graph snapshot
        facts = state.world.snapshot.by(
            In,
            lambda in1: (
                (query.dtype in in1.entity1.dtype if query.dtype else True)
                and (query.dstate is in1.entity1.dstate if query.dstate else True)
                and (in1.entity2.name == query.room if query.room else True)
            )
        )
        fact_list = [fact for fact in facts]
        print(f"--> [Found relevant facts] --> {fact_list}")

        # Synthesize concise answer grounded strictly in retrieved facts
        response = describe_facts(
            instances=fact_list,
            task_desc=f"resident question: {state.human_utterance}. Generate a short answer to this question using the facts",
            llm=llm_model
        )

        io.o.interaction(Utterance(phrase=response))
        state.human_intent = None
        state.human_utterance = None
        return state

### Interactive Queries

When running the smart home assistant script, these three components work in concert across the decision pipeline to resolve resident inquiries. You can test this in real time by launching the script alongside the simulation environment and issuing questions via the chat interface. Here is an example:

> Resident: *"Which devices are in my kitchen?"*  
> Agent: *"The kitchen contains a light device which is off and a dishwasher that is running."*

In [ ]:
!python smart_home_assistant_v5.py